# M5: дневной прогноз продаж на Kaggle

Прогнозируем продажи **30 490 пар «товар — магазин» на 28 дней**. Качество считаем по WRMSSE на 12 уровнях иерархии M5; меньше — лучше.

Текущий контроль: **LightGBM на CPU, 67 признаков, 52 × 16, 386 деревьев**. На периоде **28 марта — 24 апреля 2016 года** он получил **WRMSSE 0.53079**, Bias **−1.21%**. Это историческое окно разработки, а не серверный Kaggle score.

В ноутбуке собраны подготовка данных, принятые признаки и результаты проверенных гипотез. EWMA и короткие лаги не включены в текущую модель. Числа и сохранённые выводы взяты из выполненных запусков; при оформлении обучение не повторялось.

**Как запускать:** разделы 1–4 готовят данные и функции. Раздел 5 содержит историю результатов. В разделе 6 можно отдельно повторить CPU-контроль: по умолчанию `RUN_CONTROL_TRAINING = False`. Ночной подбор параметров будет вынесен в отдельный ноутбук.


## 1. Окружение и данные

Для запуска на Kaggle подключаем **M5 Forecasting — Accuracy** через Input. Подготовка находит папку с исходными CSV; предыдущие модели для этого не нужны.

В сохранённом контроле: Python 3.13.15, LightGBM 4.6.0, NumPy 2.1.3, pandas 2.3.3. Bottleneck отключаем до расчёта признаков: при переносе он менял результат стандартного отклонения цены. При повторении контроля версии проверяются отдельно.


In [ ]:
from contextlib import redirect_stdout
from functools import lru_cache
from pathlib import Path
from time import perf_counter
from zipfile import ZIP_DEFLATED, ZipFile
import gc
import io
import json
import os
import platform

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

HORIZON = 28
N_ORIGINS = 52
HORIZONS_PER_ORIGIN = 16
PUBLIC_ORIGIN = pd.Timestamp("2016-04-24")
CONTROL_ORIGIN = pd.Timestamp("2016-03-27")
cb_public_origin = PUBLIC_ORIGIN
RUN_CONTROL_TRAINING = False

PROJECT_ROOT = Path("/kaggle/working/demand-planner")
RESULTS = PROJECT_ROOT / "results/m5_wrmsse"
MODELS = PROJECT_ROOT / "models"
for directory in (RESULTS, MODELS):
    directory.mkdir(parents=True, exist_ok=True)

required_files = (
    "calendar.csv",
    "sales_train_validation.csv",
    "sell_prices.csv",
    "sales_train_evaluation.csv",
)
candidates = sorted({
    path.parent
    for path in Path("/kaggle/input").rglob("calendar.csv")
    if all((path.parent / filename).is_file() for filename in required_files)
})
if len(candidates) != 1:
    raise FileNotFoundError(
        "Подключи M5 Forecasting — Accuracy через Input → Add Input. "
        f"Подходящих папок: {len(candidates)}."
    )
DATA_RAW = candidates[0]

pd.set_option("compute.use_bottleneck", False)
pd.set_option("display.max_columns", 30)

BASE_FEATURE_COLUMNS = ['item_id',
 'dept_id',
 'cat_id',
 'store_id',
 'state_id',
 'mean_7d',
 'mean_28d',
 'mean_56d',
 'std_28d',
 'zero_share_28d',
 'trend_7d_28d',
 'weekday_mean_4w',
 'lag_28d',
 'lag_35d',
 'lag_42d',
 'lag_56d',
 'lag_364d',
 'horizon',
 'day_of_week',
 'day_of_month',
 'month',
 'week_of_year',
 'year',
 'snap',
 'has_event',
 'price_last_known',
 'price_observed_this_week',
 'price_change_1w',
 'price_change_4w',
 'price_vs_mean_4w',
 'price_vs_mean_13w',
 'price_vs_max_13w',
 'price_relative_std_13w',
 'price_observed_share_13w',
 'days_since_last_sale',
 'has_never_sold',
 'sale_day_share_7d',
 'sale_day_share_56d',
 'sale_frequency_change_7d_vs_prev21d',
 'mean_positive_sale_28d',
 'mean_positive_sale_56d',
 'positive_sale_cv2_56d',
 'max_daily_sale_28d',
 'store_dept_mean_7d',
 'store_dept_mean_28d',
 'store_dept_mean_56d',
 'store_dept_trend_7d_28d',
 'store_dept_trend_28d_56d',
 'store_dept_cv_28d',
 'store_dept_weekday_mean_4w',
 'store_dept_weekday_factor_4w',
 'item_share_store_dept_28d',
 'item_share_change_7d_28d',
 'other_stores_mean_7d',
 'other_stores_mean_28d',
 'other_stores_mean_56d',
 'other_stores_trend_7d_28d',
 'other_stores_trend_28d_56d',
 'other_stores_cv_28d',
 'other_stores_weekday_mean_4w',
 'other_stores_weekday_factor_4w',
 'own_vs_other_stores_mean_28d',
 'own_vs_other_stores_trend']

TARGET_PRICE_FEATURES = ['price_target_week',
 'price_target_vs_last',
 'price_target_vs_mean_13w',
 'price_target_missing']

FEATURE_COLUMNS = [*BASE_FEATURE_COLUMNS, *TARGET_PRICE_FEATURES]
CATEGORICAL_FEATURES = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']

MODEL_PARAMETERS = {'boosting_type': 'gbdt',
 'class_weight': None,
 'colsample_bytree': 0.9,
 'importance_type': 'split',
 'learning_rate': 0.05,
 'max_depth': -1,
 'min_child_samples': 100,
 'min_child_weight': 0.001,
 'min_split_gain': 0.0,
 'n_estimators': 386,
 'n_jobs': 4,
 'num_leaves': 63,
 'objective': 'tweedie',
 'random_state': 42,
 'reg_alpha': 0.0,
 'reg_lambda': 1.0,
 'subsample': 0.8,
 'subsample_for_bin': 200000,
 'subsample_freq': 1,
 'tweedie_variance_power': 1.5,
 'metric': 'None',
 'max_bin': 255,
 'device_type': 'cpu',
 'deterministic': True,
 'force_col_wise': True,
 'verbosity': -1}

EXPECTED_LIBRARIES = {'lightgbm': '4.6.0', 'numpy': '2.1.3', 'pandas': '2.3.3'}

SAVED_CONTROL_METRICS = {'WRMSSE': 0.5307854625650568, 'Bias': -0.012070512988163973}

runtime_versions = {
    "python": platform.python_version(),
    "lightgbm": lgb.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
print("Данные:", DATA_RAW)
print("Версии:", runtime_versions)
print("Признаков в контроле:", len(FEATURE_COLUMNS))
print("Повторное обучение:", RUN_CONTROL_TRAINING)


### 1.1. Календарь и периоды

В доступной истории — 1 913 дней, по 24 апреля 2016 года. Public-период уже используется при разработке. Период 23 мая — 19 июня оставляем для следующей итоговой проверки. Для каждого запуска признаки продаж ограничены его датой отсчёта.


In [4]:
calendar = pd.read_csv(DATA_RAW / "calendar.csv", parse_dates=["date"])
calendar_by_day = calendar.set_index("d")

periods = pd.DataFrame(
    [
        {"period": "История", "first_day": "d_1", "last_day": "d_1913"},
        {"period": "Public / разработка", "first_day": "d_1914", "last_day": "d_1941"},
        {
            "period": "Private / финальная проверка",
            "first_day": "d_1942",
            "last_day": "d_1969",
        },
    ]
)

periods["start"] = periods["first_day"].map(calendar_by_day["date"])
periods["end"] = periods["last_day"].map(calendar_by_day["date"])
periods["days"] = (periods["end"] - periods["start"]).dt.days + 1

assert periods.loc[periods["period"].ne("История"), "days"].eq(HORIZON).all()
display(periods.set_index("period"))


,first_day,last_day,start,end,days
period,,,,,
История,d_1,d_1913,2011-01-29,2016-04-24,1913
Public / разработка,d_1914,d_1941,2016-04-25,2016-05-22,28
Private / финальная проверка,d_1942,d_1969,2016-05-23,2016-06-19,28


### 1.2. Товары и иерархия

В данных 3 049 товаров и 10 магазинов. Из 30 490 исходных рядов строятся 42 840 рядов на 12 уровнях: от общей суммы до товара в магазине. Для всех таблиц фиксируем одинаковый порядок рядов.


In [5]:
identifier_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

series_metadata = pd.read_csv(
    DATA_RAW / "sales_train_validation.csv",
    usecols=identifier_columns,
    dtype="string",
)

assert not series_metadata.duplicated(["item_id", "store_id"]).any()
assert len(series_metadata) == 30_490
assert series_metadata["store_id"].nunique() == 10

M5_LEVELS = {
    "Total": [],
    "State": ["state_id"],
    "Store": ["store_id"],
    "Category": ["cat_id"],
    "Department": ["dept_id"],
    "State / Category": ["state_id", "cat_id"],
    "State / Department": ["state_id", "dept_id"],
    "Store / Category": ["store_id", "cat_id"],
    "Store / Department": ["store_id", "dept_id"],
    "Item": ["item_id"],
    "Item / State": ["item_id", "state_id"],
    "Item / Store": ["item_id", "store_id"],
}

hierarchy = pd.DataFrame(
    [
        {
            "level": level,
            "series": (
                series_metadata.groupby(columns, observed=True).ngroups
                if columns
                else 1
            ),
        }
        for level, columns in M5_LEVELS.items()
    ]
).set_index("level")

assert int(hierarchy["series"].sum()) == 42_840
display(hierarchy)
print(f"Всего оцениваемых рядов: {hierarchy['series'].sum():,}")


,series
level,
Total,1
State,3
Store,10
Category,3
Department,7
State / Category,9
State / Department,21
Store / Category,30
Store / Department,70


Всего оцениваемых рядов: 42,840


### 1.3. Продажи и денежные веса

Читаем дневные продажи из `sales_train_validation.csv`. Для расчёта денежных весов берём продажи и цены последних 28 доступных дней. В исторических окнах этот период заканчивается на соответствующей дате отсчёта.


In [6]:
M5_KEYS = ["store_id", "item_id"]
TRAIN_END_DAY = 1913

train_day_columns = [f"d_{day}" for day in range(1, TRAIN_END_DAY + 1)]

series_index = pd.MultiIndex.from_frame(series_metadata[M5_KEYS])

train_daily = pd.read_csv(
    DATA_RAW / "sales_train_validation.csv",
    usecols=[*M5_KEYS, *train_day_columns],
    dtype={
        "store_id": "string",
        "item_id": "string",
        **{column: "int32" for column in train_day_columns},
    },
)

train_daily = (
    train_daily.set_index(M5_KEYS).reindex(series_index).loc[:, train_day_columns]
)

train_daily.columns = pd.DatetimeIndex(
    calendar.set_index("d").loc[train_day_columns, "date"],
    name="date",
)

weight_dates = train_daily.columns[-28:]

weight_weeks = calendar.set_index("date").loc[weight_dates, "wm_yr_wk"].unique()

price_parts = []

for chunk in pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
    dtype={
        "store_id": "string",
        "item_id": "string",
        "wm_yr_wk": "int32",
        "sell_price": "float64",
    },
    chunksize=200_000,
):
    selected = chunk.loc[chunk["wm_yr_wk"].isin(weight_weeks)]

    if not selected.empty:
        price_parts.append(selected.copy())

prices_for_weights = pd.concat(
    price_parts,
    ignore_index=True,
)

del price_parts

print(f"Рядов: {len(train_daily):,}")
print(f"Дней истории: {train_daily.shape[1]:,}")
print(
    "Период для весов:",
    weight_dates.min().date(),
    "—",
    weight_dates.max().date(),
)


Рядов: 30,490
Дней истории: 1,913
Период для весов: 2016-03-28 — 2016-04-24


## 2. Метрика WRMSSE

Для каждого ряда средний квадрат ошибки за 28 дней нормируется на средний квадрат прошлых изменений продаж. Историю до первой ненулевой продажи исключаем из масштаба. Затем ошибки взвешиваются по денежному объёму последних 28 дней; каждый из 12 уровней получает одинаковый общий вес.

$$
\mathrm{WRMSSE} = \sum_i w_i
\sqrt{\frac{\frac{1}{28}\sum_{h=1}^{28}(y_{i,h}-\hat y_{i,h})^2}{s_i}}
$$

Проверяем индексы, даты, пропуски, масштабы и сумму весов. Формула следует [описанию M5](https://www.kaggle.com/competitions/m5-forecasting-accuracy/overview/evaluation). Независимый пересчёт ранее совпал по рабочей проверке, но его отдельный численный отчёт в этом экспорте отсутствует.


In [7]:
from scipy.sparse import csr_matrix


def calculate_rmsse_scale(history_values, aggregation, block_size=128):
    n_groups = aggregation.shape[0]
    n_days = history_values.shape[1]

    first_nonzero = np.full(n_groups, n_days, dtype=np.int32)
    squared_difference_sum = np.zeros(n_groups, dtype=np.float64)
    previous = np.zeros(n_groups, dtype=np.float64)

    for start in range(0, n_days, block_size):
        stop = min(start + block_size, n_days)

        block = np.asarray(
            aggregation @ history_values[:, start:stop],
            dtype=np.float64,
        )

        positive = block > 0
        newly_active = (first_nonzero == n_days) & positive.any(axis=1)

        if newly_active.any():
            first_nonzero[newly_active] = start + positive[newly_active].argmax(axis=1)

        differences = np.empty_like(block)
        differences[:, 0] = block[:, 0] - previous
        differences[:, 1:] = np.diff(block, axis=1)

        positions = np.arange(start, stop)
        exclude = positions[None, :] <= first_nonzero[:, None]
        differences[exclude] = 0.0

        squared_difference_sum += np.einsum(
            "ij,ij->i",
            differences,
            differences,
        )

        previous = block[:, -1].copy()

    difference_count = n_days - first_nonzero - 1

    return np.divide(
        squared_difference_sum,
        difference_count,
        out=np.zeros(n_groups, dtype=np.float64),
        where=difference_count > 0,
    )


class M5WRMSSE:
    def __init__(
        self,
        history,
        metadata,
        calendar,
        prices,
        levels,
        horizon=28,
    ):
        if len(metadata) != 30_490 or len(levels) != 12:
            raise ValueError("Для полного M5 нужны 30 490 рядов и 12 уровней.")

        self.series_index = pd.MultiIndex.from_frame(metadata[M5_KEYS])

        if (
            self.series_index.has_duplicates
            or history.index.has_duplicates
            or len(history) != len(self.series_index)
            or not self.series_index.isin(history.index).all()
        ):
            raise ValueError("Идентификаторы истории и metadata не совпадают.")

        history = history.reindex(self.series_index)

        if history.shape[1] < 28:
            raise ValueError("Для расчёта весов нужны минимум 28 дней.")

        if not isinstance(history.columns, pd.DatetimeIndex):
            raise TypeError("Столбцы истории должны содержать даты.")

        expected_history_dates = pd.date_range(
            history.columns[0],
            periods=history.shape[1],
            freq="D",
        )

        if not history.columns.equals(expected_history_dates):
            raise ValueError("В истории есть пропуски или нарушен порядок дней.")

        history_values = history.to_numpy(copy=False)

        if not np.isfinite(history_values).all() or (history_values < 0).any():
            raise ValueError(
                "История должна содержать конечные неотрицательные продажи."
            )

        self.horizon = horizon
        self.n_levels = len(levels)
        self.forecast_dates = pd.date_range(
            history.columns[-1] + pd.Timedelta(days=1),
            periods=horizon,
            freq="D",
            name="date",
        )

        recent_dates = history.columns[-28:]

        week_codes = (
            calendar.set_index("date").reindex(recent_dates)["wm_yr_wk"].to_numpy()
        )

        if pd.isna(week_codes).any():
            raise ValueError("В календаре отсутствуют даты для расчёта весов.")

        known_prices = prices["sell_price"].dropna().to_numpy()

        if not np.isfinite(known_prices).all() or (known_prices <= 0).any():
            raise ValueError("Известные цены должны быть конечными и положительными.")

        price_table = prices.pivot(
            index=M5_KEYS,
            columns="wm_yr_wk",
            values="sell_price",
        )

        daily_prices = price_table.reindex(
            index=self.series_index,
            columns=week_codes,
        ).to_numpy(dtype=np.float64)

        recent_units = history.iloc[:, -28:].to_numpy(dtype=np.float64)

        missing_price_with_sales = (recent_units > 0) & ~np.isfinite(daily_prices)

        if missing_price_with_sales.any():
            raise ValueError(
                "Есть положительные продажи без цены: веса нельзя посчитать корректно."
            )

        dollar_sales = (recent_units * np.nan_to_num(daily_prices, nan=0.0)).sum(axis=1)

        total_dollar_sales = dollar_sales.sum()

        if total_dollar_sales <= 0:
            raise ValueError("Суммарные денежные продажи равны нулю.")

        metadata = metadata.reset_index(drop=True)

        self.level_specs = {}
        level_rows = []

        for level_name, columns in levels.items():
            if columns:
                group_index = pd.MultiIndex.from_frame(metadata[columns])
                codes, groups = pd.factorize(
                    group_index,
                    sort=True,
                )
                n_groups = len(groups)
            else:
                codes = np.zeros(len(metadata), dtype=np.int32)
                n_groups = 1

            aggregation = csr_matrix(
                (
                    np.ones(len(metadata), dtype=np.float64),
                    (codes, np.arange(len(metadata))),
                ),
                shape=(n_groups, len(metadata)),
            )

            print(f"Масштабы: {level_name} — {n_groups:,} рядов")

            scale = calculate_rmsse_scale(
                history_values,
                aggregation,
            )

            group_dollar_sales = np.asarray(aggregation @ dollar_sales).ravel()

            weights = group_dollar_sales / total_dollar_sales / self.n_levels

            if ((scale <= 0) & (weights > 0)).any():
                raise ValueError(
                    f"{level_name}: есть ряд с положительным весом "
                    "и нулевым масштабом RMSSE."
                )

            self.level_specs[level_name] = {
                "aggregation": aggregation,
                "scale": scale,
                "weights": weights,
            }

            level_rows.append(
                {
                    "level": level_name,
                    "series": n_groups,
                    "weight_sum": weights.sum(),
                }
            )

        self.level_info = pd.DataFrame(level_rows).set_index("level")

        if self.level_info["series"].sum() != 42_840:
            raise ValueError("Получено неверное число рядов иерархии.")

        if not np.isclose(self.level_info["weight_sum"].sum(), 1.0):
            raise ValueError("Сумма весов WRMSSE отличается от единицы.")

    def _aligned_values(self, frame):
        if not isinstance(frame, pd.DataFrame):
            raise TypeError("Передай DataFrame с идентификаторами и датами.")

        if (
            frame.index.has_duplicates
            or len(frame) != len(self.series_index)
            or not self.series_index.isin(frame.index).all()
        ):
            raise ValueError("Набор товаров и магазинов не совпадает.")

        if not frame.columns.equals(self.forecast_dates):
            raise ValueError(
                "Столбцы должны содержать следующие 28 дат " "в правильном порядке."
            )

        values = frame.reindex(self.series_index).to_numpy(dtype=np.float64)

        if not np.isfinite(values).all():
            raise ValueError("В фактах или прогнозах есть NaN либо бесконечность.")

        return values

    def score(self, actual, prediction):
        actual_values = self._aligned_values(actual)
        prediction_values = self._aligned_values(prediction)

        bottom_errors = prediction_values - actual_values
        rows = []

        for level_name, spec in self.level_specs.items():
            aggregated_errors = spec["aggregation"] @ bottom_errors

            mse = np.mean(
                np.square(aggregated_errors),
                axis=1,
            )

            rmsse = np.sqrt(
                np.divide(
                    mse,
                    spec["scale"],
                    out=np.zeros_like(mse),
                    where=spec["scale"] > 0,
                )
            )

            contribution = float(np.dot(spec["weights"], rmsse))

            rows.append(
                {
                    "level": level_name,
                    "weighted_RMSSE": contribution * self.n_levels,
                    "contribution": contribution,
                }
            )

        report = pd.DataFrame(rows).set_index("level")
        overall = float(report["contribution"].sum())

        return overall, report


### 2.1. Оценщик для конца доступной истории

Создаём оценщик до 24 апреля и проверяем сумму весов и размер иерархии. Для исторического контроля ниже будет создан отдельный оценщик до 27 марта. Сохранённый вывод относится к предыдущему запуску.


In [8]:
wrmsse_evaluator = M5WRMSSE(
    history=train_daily,
    metadata=series_metadata,
    calendar=calendar,
    prices=prices_for_weights,
    levels=M5_LEVELS,
    horizon=HORIZON,
)

display(
    wrmsse_evaluator.level_info.style.format(
        {
            "series": "{:,.0f}",
            "weight_sum": "{:.6f}",
        }
    )
)

print(
    "Всего рядов:",
    f"{wrmsse_evaluator.level_info['series'].sum():,}",
)

print(
    "Сумма весов:",
    f"{wrmsse_evaluator.level_info['weight_sum'].sum():.6f}",
)

print(
    "Даты будущего прогноза:",
    wrmsse_evaluator.forecast_dates.min().date(),
    "—",
    wrmsse_evaluator.forecast_dates.max().date(),
)


Масштабы: Total — 1 рядов
Масштабы: State — 3 рядов
Масштабы: Store — 10 рядов
Масштабы: Category — 3 рядов
Масштабы: Department — 7 рядов
Масштабы: State / Category — 9 рядов
Масштабы: State / Department — 21 рядов
Масштабы: Store / Category — 30 рядов
Масштабы: Store / Department — 70 рядов
Масштабы: Item — 3,049 рядов
Масштабы: Item / State — 9,147 рядов
Масштабы: Item / Store — 30,490 рядов


,series,weight_sum
level,,
Total,1,0.083333
State,3,0.083333
Store,10,0.083333
Category,3,0.083333
Department,7,0.083333
State / Category,9,0.083333
State / Department,21,0.083333
Store / Category,30,0.083333
Store / Department,70,0.083333


Всего рядов: 42,840
Сумма весов: 1.000000
Даты будущего прогноза: 2016-04-25 — 2016-05-22


### 2.2. WRMSSE и суммарный Bias

`price_evaluate` возвращает WRMSSE, отчёт по уровням и Bias: сумма прогноза, делённая на сумму факта, минус один. Отрицательный Bias означает недопрогноз. Коэффициент калибровки позже будем выбирать по WRMSSE; сам по себе нулевой Bias не гарантирует лучшего прогноза.


In [12]:
def price_evaluate(evaluator, actual, prediction):
    score, report = evaluator.score(actual, prediction)
    actual_total = (
        actual.reindex(index=prediction.index, columns=prediction.columns)
        .to_numpy(dtype=np.float64)
        .sum()
    )
    metrics = {
        "WRMSSE": float(score),
        "Bias": float(prediction.to_numpy().sum() / actual_total - 1),
    }
    return (metrics, report)


## 3. Признаки

Итоговые 67 столбцов собраны в `FEATURE_COLUMNS`: 63 принятых ранее и четыре признака цены целевой недели. Пять идентификаторов передаются как категории. Продажи для признаков берём только до даты отсчёта; календарь и предоставленные цены целевой недели считаем известными заранее.

### 3.1. Продажи и календарь

Средние за 7, 28 и 56 дней, разброс, доля нулей, изменение уровня, среднее нужного дня недели и лаги 28–364 дня. Лаги от 28 дней доступны для всего горизонта. Календарные признаки и SNAP относятся к целевому дню.


In [9]:
daily_categorical_features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
]

daily_history = train_daily.reindex(wrmsse_evaluator.series_index)

daily_history_dates = daily_history.columns
daily_history_values = daily_history.to_numpy(copy=False)

daily_identifiers = (
    series_metadata.set_index(M5_KEYS)
    .loc[wrmsse_evaluator.series_index]
    .reset_index()
    .loc[:, daily_categorical_features]
)

daily_identifiers.index = wrmsse_evaluator.series_index

for column in daily_categorical_features:
    daily_identifiers[column] = daily_identifiers[column].astype("category")

daily_calendar = calendar.set_index("date")


def make_daily_features(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)

    horizon = (target_date - forecast_origin).days

    if not 1 <= horizon <= HORIZON:
        raise ValueError("Горизонт должен быть от 1 до 28 дней.")

    origin_position = daily_history_dates.get_indexer([forecast_origin])[0]

    if origin_position < 55:
        raise ValueError(
            "Дата прогноза должна присутствовать в истории "
            "и иметь минимум 56 доступных дней."
        )

    X = daily_identifiers.copy()
    known_history = daily_history_values[:, : origin_position + 1]

    for window in (7, 28, 56):
        X[f"mean_{window}d"] = known_history[:, -window:].mean(axis=1).astype("float32")

    recent_28 = known_history[:, -28:]

    X["std_28d"] = recent_28.std(axis=1, ddof=1).astype("float32")

    X["zero_share_28d"] = (recent_28 == 0).mean(axis=1).astype("float32")

    X["trend_7d_28d"] = (X["mean_7d"] - X["mean_28d"]).astype("float32")

    recent_dates = daily_history_dates[origin_position - 27 : origin_position + 1]

    weekday_mask = recent_dates.dayofweek == target_date.dayofweek

    X["weekday_mean_4w"] = recent_28[:, weekday_mask].mean(axis=1).astype("float32")

    for lag in (28, 35, 42, 56, 364):
        source_date = target_date - pd.Timedelta(days=lag)

        if source_date > forecast_origin:
            raise ValueError("Лаг обращается к неизвестному будущему.")

        source_position = daily_history_dates.get_indexer([source_date])[0]

        if source_position >= 0:
            X[f"lag_{lag}d"] = daily_history_values[:, source_position].astype(
                "float32"
            )
        else:
            X[f"lag_{lag}d"] = np.float32(np.nan)

    calendar_values = daily_calendar.loc[target_date]

    target_features = {
        "horizon": horizon,
        "day_of_week": target_date.dayofweek,
        "day_of_month": target_date.day,
        "month": target_date.month,
        "week_of_year": int(target_date.isocalendar().week),
        "year": target_date.year,
    }

    for name, value in target_features.items():
        X[name] = np.int16(value)

    snap_by_state = {
        state: int(calendar_values[f"snap_{state}"]) for state in ("CA", "TX", "WI")
    }

    X["snap"] = X["state_id"].map(snap_by_state).astype("int8")

    X["has_event"] = np.int8(
        pd.notna(calendar_values["event_name_1"])
        or pd.notna(calendar_values["event_name_2"])
    )

    return X


### 3.2. История цены

Последняя известная цена, изменения за 1 и 4 недели, положение относительно истории и частота наблюдения цены. Эти статистики заканчиваются на дате отсчёта. Пропуск цены не интерпретируем как подтверждённое отсутствие товара на складе.


In [11]:
price_data = pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    dtype={
        "store_id": "category",
        "item_id": "category",
        "wm_yr_wk": "int32",
        "sell_price": "float64",
    },
)
price_week_starts = calendar.groupby("wm_yr_wk")["date"].min().sort_values()
price_week_order = price_week_starts.loc[price_week_starts <= cb_public_origin].index
price_data = price_data.loc[price_data["wm_yr_wk"].isin(price_week_order)].copy()
price_week_table = (
    price_data.pivot(index=M5_KEYS, columns="wm_yr_wk", values="sell_price")
    .reindex(index=wrmsse_evaluator.series_index, columns=price_week_order)
    .astype("float32")
)
price_last_known_table = price_week_table.ffill(axis=1)


@lru_cache(maxsize=64)
def price_features_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    week = int(daily_calendar.loc[forecast_origin, "wm_yr_wk"])
    position = price_week_table.columns.get_loc(week)
    if position < 12:
        raise ValueError("Для признаков цены нужны 13 недель истории.")
    recent_4 = price_week_table.iloc[:, position - 3 : position + 1]
    recent_13 = price_week_table.iloc[:, position - 12 : position + 1]
    current = price_last_known_table.iloc[:, position]
    previous_1 = price_last_known_table.iloc[:, position - 1]
    previous_4 = price_last_known_table.iloc[:, position - 4]
    mean_4 = recent_4.mean(axis=1)
    mean_13 = recent_13.mean(axis=1)
    return pd.DataFrame(
        {
            "price_last_known": current,
            "price_observed_this_week": price_week_table.iloc[:, position].notna(),
            "price_change_1w": current / previous_1 - 1,
            "price_change_4w": current / previous_4 - 1,
            "price_vs_mean_4w": current / mean_4 - 1,
            "price_vs_mean_13w": current / mean_13 - 1,
            "price_vs_max_13w": current / recent_13.max(axis=1) - 1,
            "price_relative_std_13w": recent_13.std(axis=1, ddof=0) / mean_13,
            "price_observed_share_13w": recent_13.notna().mean(axis=1),
        }
    ).astype("float32")


def make_daily_features_with_price(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    base = make_daily_features(forecast_origin=forecast_origin, target_date=target_date)
    return base.join(price_features_at_origin(forecast_origin), validate="one_to_one")


### 3.3. Регулярность продаж

Сколько дней прошло с последней продажи, как часто бывают продажи, как изменилась их частота и сколько продаётся в ненулевой день. Эти признаки описывают нерегулярный спрос; причины нулевых продаж из них однозначно не следуют.


In [14]:
@lru_cache(maxsize=64)
def regularity_features_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError(
            "Дата должна присутствовать в истории и иметь минимум 56 доступных дней."
        )
    known = daily_history_values[:, : position + 1]
    positive_history = known > 0
    has_ever_sold = positive_history.any(axis=1)
    days_since_sale = positive_history[:, ::-1].argmax(axis=1).astype("float32")
    days_since_sale[~has_ever_sold] = np.nan
    recent_56 = known[:, -56:].astype("float64")
    recent_28 = recent_56[:, -28:]
    positive_56 = positive_history[:, -56:]
    positive_28 = positive_56[:, -28:]
    count_56 = positive_56.sum(axis=1)
    count_28 = positive_28.sum(axis=1)
    sum_56 = recent_56.sum(axis=1)
    sum_28 = recent_28.sum(axis=1)
    mean_positive_56 = np.divide(
        sum_56, count_56, out=np.full(len(known), np.nan), where=count_56 > 0
    )
    mean_positive_28 = np.divide(
        sum_28, count_28, out=np.full(len(known), np.nan), where=count_28 > 0
    )
    squared_sum_56 = np.einsum("ij,ij->i", recent_56, recent_56)
    correction = np.divide(
        np.square(sum_56), count_56, out=np.zeros(len(known)), where=count_56 > 0
    )
    positive_variance_56 = np.divide(
        np.maximum(squared_sum_56 - correction, 0.0),
        count_56 - 1,
        out=np.full(len(known), np.nan),
        where=count_56 > 1,
    )
    positive_cv2_56 = np.divide(
        positive_variance_56,
        np.square(mean_positive_56),
        out=np.full(len(known), np.nan),
        where=count_56 > 1,
    )
    frequency_7 = positive_56[:, -7:].mean(axis=1)
    frequency_previous_21 = positive_56[:, -28:-7].mean(axis=1)
    return pd.DataFrame(
        {
            "days_since_last_sale": days_since_sale,
            "has_never_sold": ~has_ever_sold,
            "sale_day_share_7d": frequency_7,
            "sale_day_share_56d": positive_56.mean(axis=1),
            "sale_frequency_change_7d_vs_prev21d": frequency_7 - frequency_previous_21,
            "mean_positive_sale_28d": mean_positive_28,
            "mean_positive_sale_56d": mean_positive_56,
            "positive_sale_cv2_56d": positive_cv2_56,
            "max_daily_sale_28d": recent_28.max(axis=1),
        },
        index=wrmsse_evaluator.series_index,
    ).astype("float32")


def make_daily_features_with_regularity(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    X = make_daily_features_with_price(forecast_origin, target_date)
    return X.join(regularity_features_at_origin(forecast_origin), validate="one_to_one")


### 3.4. Отдел в магазине

Агрегируем историю отдела: уровни, тренды, разброс и профиль дня недели. Для отдельного товара добавляем его долю в продажах отдела. Агрегация не обращается к будущим продажам.


In [16]:
group_aggregation = wrmsse_evaluator.level_specs["Store / Department"]["aggregation"]
group_membership = group_aggregation.tocoo()
group_codes = np.empty(len(wrmsse_evaluator.series_index), dtype=np.int32)
group_codes[group_membership.col] = group_membership.row
group_daily_sales = np.asarray(
    group_aggregation @ daily_history_values, dtype=np.float64
)


def group_safe_ratio(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)
    return np.divide(
        numerator,
        denominator,
        out=np.full(numerator.shape, np.nan),
        where=denominator > 0,
    )


@lru_cache(maxsize=64)
def group_statistics_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError("Для групповых признаков нужны 56 доступных дней.")
    known = group_daily_sales[:, : position + 1]
    recent_28 = known[:, -28:]
    recent_dates = daily_history_dates[position - 27 : position + 1]
    statistics = {
        f"mean_{window}d": known[:, -window:].mean(axis=1) for window in (7, 28, 56)
    }
    statistics["std_28d"] = recent_28.std(axis=1, ddof=1)
    statistics["weekday_mean_4w"] = np.column_stack(
        [
            recent_28[:, recent_dates.dayofweek == weekday].mean(axis=1)
            for weekday in range(7)
        ]
    )
    return statistics


def make_daily_features_with_groups(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)
    X = make_daily_features_with_regularity(forecast_origin, target_date)
    statistics = group_statistics_at_origin(forecast_origin)
    mean_7 = statistics["mean_7d"][group_codes]
    mean_28 = statistics["mean_28d"][group_codes]
    mean_56 = statistics["mean_56d"][group_codes]
    weekday_mean = statistics["weekday_mean_4w"][group_codes, target_date.dayofweek]
    item_share_7 = group_safe_ratio(X["mean_7d"].to_numpy(), mean_7)
    item_share_28 = group_safe_ratio(X["mean_28d"].to_numpy(), mean_28)
    added_features = {
        "store_dept_mean_7d": mean_7,
        "store_dept_mean_28d": mean_28,
        "store_dept_mean_56d": mean_56,
        "store_dept_trend_7d_28d": group_safe_ratio(mean_7, mean_28) - 1,
        "store_dept_trend_28d_56d": group_safe_ratio(mean_28, mean_56) - 1,
        "store_dept_cv_28d": group_safe_ratio(
            statistics["std_28d"][group_codes], mean_28
        ),
        "store_dept_weekday_mean_4w": weekday_mean,
        "store_dept_weekday_factor_4w": group_safe_ratio(weekday_mean, mean_28),
        "item_share_store_dept_28d": item_share_28,
        "item_share_change_7d_28d": item_share_7 - item_share_28,
    }
    for column, values in added_features.items():
        X[column] = values.astype("float32")
    return X


### 3.5. Тот же товар в других магазинах

Вычитаем текущий магазин из продаж товара по сети и делим на число остальных магазинов. По этой истории считаем уровни, тренды и профиль дня недели. После блока получаем прежние 63 признака.


In [17]:
cross_store_aggregation = wrmsse_evaluator.level_specs["Item"]["aggregation"]
cross_store_membership = cross_store_aggregation.tocoo()
cross_store_codes = np.empty(len(wrmsse_evaluator.series_index), dtype=np.int32)
cross_store_codes[cross_store_membership.col] = cross_store_membership.row
cross_store_counts = np.asarray(cross_store_aggregation.sum(axis=1)).ravel()
cross_store_other_counts = cross_store_counts[cross_store_codes] - 1
cross_store_daily_totals = np.asarray(
    cross_store_aggregation @ daily_history_values, dtype=np.float64
)


@lru_cache(maxsize=64)
def cross_store_statistics_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError("Для признаков других магазинов нужны 56 дней истории.")
    start = position - 55
    network_sales = cross_store_daily_totals[cross_store_codes, start : position + 1]
    own_sales = daily_history_values[:, start : position + 1]
    other_sales = (network_sales - own_sales) / cross_store_other_counts[:, None]
    recent_28 = other_sales[:, -28:]
    recent_dates = daily_history_dates[position - 27 : position + 1]
    statistics = {
        f"mean_{window}d": other_sales[:, -window:].mean(axis=1).astype("float32")
        for window in (7, 28, 56)
    }
    statistics["std_28d"] = recent_28.std(axis=1, ddof=1).astype("float32")
    statistics["weekday_mean_4w"] = np.column_stack(
        [
            recent_28[:, recent_dates.dayofweek == weekday].mean(axis=1)
            for weekday in range(7)
        ]
    ).astype("float32")
    return statistics


def make_daily_features_with_cross_store(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)
    X = make_daily_features_with_groups(forecast_origin, target_date)
    statistics = cross_store_statistics_at_origin(forecast_origin)
    mean_7 = statistics["mean_7d"]
    mean_28 = statistics["mean_28d"]
    mean_56 = statistics["mean_56d"]
    weekday_mean = statistics["weekday_mean_4w"][:, target_date.dayofweek]
    own_trend = group_safe_ratio(X["mean_7d"].to_numpy(), X["mean_28d"].to_numpy()) - 1
    other_trend = group_safe_ratio(mean_7, mean_28) - 1
    added_features = {
        "other_stores_mean_7d": mean_7,
        "other_stores_mean_28d": mean_28,
        "other_stores_mean_56d": mean_56,
        "other_stores_trend_7d_28d": other_trend,
        "other_stores_trend_28d_56d": group_safe_ratio(mean_28, mean_56) - 1,
        "other_stores_cv_28d": group_safe_ratio(statistics["std_28d"], mean_28),
        "other_stores_weekday_mean_4w": weekday_mean,
        "other_stores_weekday_factor_4w": group_safe_ratio(weekday_mean, mean_28),
        "own_vs_other_stores_mean_28d": group_safe_ratio(
            X["mean_28d"].to_numpy(), mean_28
        ),
        "own_vs_other_stores_trend": own_trend - other_trend,
    }
    for column, values in added_features.items():
        X[column] = values.astype("float32")
    return X


### 3.6. Цена целевой недели: ещё четыре признака

Добавляем `price_target_week`, изменения относительно последней известной и средней исторической цены, а также `price_target_missing`. Получаем **67 признаков**. Средняя историческая цена считается за 13 недель до даты отсчёта включительно.

Цены читаем из полного `sell_prices.csv`, поскольку таблица исторических цен ограничена концом доступной истории. В прикладном прогнозе эти признаки допустимы, только если будущий ценовой план действительно известен. EWMA и короткие лаги в эту функцию не входят.


In [ ]:
CATEGORY_TYPES = {
    column: pd.CategoricalDtype(
        categories=daily_identifiers[column].cat.categories,
        ordered=False,
    )
    for column in CATEGORICAL_FEATURES
}

target_price_source = pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
    dtype={
        "store_id": "category",
        "item_id": "category",
        "wm_yr_wk": "int32",
        "sell_price": "float32",
    },
)
observed_prices = target_price_source["sell_price"].dropna().to_numpy()
if not np.isfinite(observed_prices).all() or (observed_prices <= 0).any():
    raise ValueError("Некорректные наблюдаемые цены.")
target_week_prices = (
    target_price_source.pivot(
        index=M5_KEYS, columns="wm_yr_wk", values="sell_price"
    )
    .reindex(index=wrmsse_evaluator.series_index)
    .astype("float32")
)
del target_price_source, observed_prices
gc.collect()


def prepare_categories(frame):
    for column, dtype in CATEGORY_TYPES.items():
        frame[column] = frame[column].astype(dtype)
        if frame[column].isna().any():
            raise ValueError(f"Неизвестная категория в {column}.")
    return frame


def relative_price_change(numerator, denominator):
    result = np.full(len(numerator), np.nan, dtype=np.float32)
    valid = (
        np.isfinite(numerator)
        & np.isfinite(denominator)
        & (denominator > 0)
    )
    np.divide(numerator, denominator, out=result, where=valid)
    return result - np.float32(1)


@lru_cache(maxsize=64)
def historical_price_mean_13w(forecast_origin):
    week = int(daily_calendar.loc[forecast_origin, "wm_yr_wk"])
    position = price_week_table.columns.get_loc(week)
    if position < 12:
        raise ValueError("Для среднего цены нужны 13 недель истории.")
    return (
        price_week_table.iloc[:, position - 12 : position + 1]
        .mean(axis=1)
        .to_numpy(dtype=np.float32)
    )


def make_daily_features_final(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)
    frame = make_daily_features_with_cross_store(forecast_origin, target_date)
    if (
        frame.columns.tolist() != BASE_FEATURE_COLUMNS
        or not frame.index.equals(wrmsse_evaluator.series_index)
    ):
        raise ValueError("Изменились базовые признаки или порядок рядов.")
    target_week = int(daily_calendar.loc[target_date, "wm_yr_wk"])
    target_price = target_week_prices[target_week].to_numpy()
    last_price = frame["price_last_known"].to_numpy()
    frame["price_target_week"] = target_price
    frame["price_target_vs_last"] = relative_price_change(target_price, last_price)
    frame["price_target_vs_mean_13w"] = relative_price_change(
        target_price, historical_price_mean_13w(forecast_origin)
    )
    frame["price_target_missing"] = (~np.isfinite(target_price)).astype("float32")
    return prepare_categories(frame.loc[:, FEATURE_COLUMNS])


## 4. Обучающие даты и сборка таблицы

Используем 52 еженедельные даты отсчёта, на каждой — 16 из 28 горизонтов и все 30 490 рядов. Получается **25 367 680 строк**. План расширяет прежние 52 × 2 → 4 → 8 → 16, сохраняя ранее выбранные примеры. Порядок строк совпадает с сохранённым планом контроля.

Массивы под каждый столбец выделяются заранее: одновременно хранить сотни промежуточных DataFrame не требуется. Категории и числовые типы сохраняются. Таргет не может выходить за конец обучающей истории.


In [ ]:
def make_training_plan(label_cutoff, horizons_per_origin=16):
    if horizons_per_origin not in (2, 4, 8, 16, 28):
        raise ValueError("Допустимы планы 52 × 2, 4, 8, 16 или 28.")
    label_cutoff = pd.Timestamp(label_cutoff)
    origins = pd.date_range(
        end=label_cutoff - pd.Timedelta(days=HORIZON),
        periods=N_ORIGINS,
        freq="7D",
    )
    rows = []
    for number, origin in enumerate(origins):
        first = 5 * number % HORIZON + 1
        old = {first, (5 * number + HORIZON // 2) % HORIZON + 1}
        if number == N_ORIGINS - 1:
            old = {1, HORIZON}
        four = old | (
            {10, 19}
            if number == N_ORIGINS - 1
            else {(first - 1 + shift) % HORIZON + 1 for shift in (7, 21)}
        )
        eight = four | (
            {5, 14, 23, 26}
            if number == N_ORIGINS - 1
            else {
                (first - 1 + shift) % HORIZON + 1
                for shift in (3, 10, 17, 24)
            }
        )
        if len(old) != 2 or len(four) != 4 or len(eight) != 8:
            raise ValueError("Изменился исходный план горизонтов.")
        horizons = set({2: old, 4: four}.get(horizons_per_origin, eight))
        while len(horizons) < horizons_per_origin:
            candidates = set(range(1, HORIZON + 1)) - horizons

            def separation(candidate):
                return min(
                    min(abs(candidate - old), HORIZON - abs(candidate - old))
                    for old in horizons
                )

            selected = max(
                candidates,
                key=lambda candidate: (separation(candidate), -candidate),
            )
            horizons.add(selected)
        for horizon in sorted(horizons):
            target_date = origin + pd.Timedelta(days=horizon)
            if target_date > label_cutoff:
                raise ValueError("Цель выходит за конец обучающей истории.")
            rows.append(
                {
                    "origin": origin,
                    "horizon": horizon,
                    "target_date": target_date,
                    "present_in_52x2": horizon in old,
                    "present_in_52x4": horizon in four,
                    "present_in_52x8": horizon in eight,
                }
            )
    return pd.DataFrame(rows)


def clear_feature_caches():
    for function in (
        price_features_at_origin,
        regularity_features_at_origin,
        group_statistics_at_origin,
        cross_store_statistics_at_origin,
        historical_price_mean_13w,
    ):
        function.cache_clear()
    gc.collect()


def build_feature_dataset(pairs, label_cutoff=None):
    pairs = list(pairs)
    if not pairs:
        raise ValueError("Пустой план подготовки данных.")
    n_series = len(wrmsse_evaluator.series_index)
    n_rows = len(pairs) * n_series
    storage = None
    y = np.empty(n_rows, dtype=np.float32) if label_cutoff is not None else None
    if label_cutoff is not None:
        label_cutoff = pd.Timestamp(label_cutoff)
    for number, (origin, target_date) in enumerate(pairs, start=1):
        origin = pd.Timestamp(origin)
        target_date = pd.Timestamp(target_date)
        frame = make_daily_features_final(origin, target_date)
        if storage is None:
            storage = {
                column: np.empty(
                    n_rows,
                    dtype=(
                        frame[column].cat.codes.dtype
                        if column in CATEGORICAL_FEATURES
                        else frame[column].dtype
                    ),
                )
                for column in FEATURE_COLUMNS
            }
        start = (number - 1) * n_series
        stop = start + n_series
        for column in FEATURE_COLUMNS:
            values = (
                frame[column].cat.codes.to_numpy()
                if column in CATEGORICAL_FEATURES
                else frame[column].to_numpy()
            )
            storage[column][start:stop] = values
        if y is not None:
            position = daily_history_dates.get_indexer([target_date])[0]
            if (
                position < 0
                or target_date > label_cutoff
                or origin >= target_date
                or origin > label_cutoff
            ):
                raise ValueError("Недоступная дата обучающей цели.")
            y[start:stop] = daily_history_values[:, position]
            if number % (HORIZONS_PER_ORIGIN * 13) == 0:
                print(
                    f"Подготовлено дат: {number // HORIZONS_PER_ORIGIN}/52",
                    flush=True,
                )
    if y is not None and (not np.isfinite(y).all() or (y < 0).any()):
        raise ValueError("Некорректные целевые продажи.")
    data = {
        column: (
            pd.Categorical.from_codes(storage[column], dtype=CATEGORY_TYPES[column])
            if column in CATEGORICAL_FEATURES
            else storage[column]
        )
        for column in FEATURE_COLUMNS
    }
    return pd.DataFrame(data, columns=FEATURE_COLUMNS, copy=False), y


def make_window_evaluator(origin):
    history = daily_history.loc[:, :origin]
    weeks = daily_calendar.loc[history.columns[-28:], "wm_yr_wk"].unique()
    with redirect_stdout(io.StringIO()):
        evaluator = M5WRMSSE(
            history=history,
            metadata=series_metadata,
            calendar=calendar,
            prices=price_data.loc[price_data["wm_yr_wk"].isin(weeks)],
            levels=M5_LEVELS,
            horizon=HORIZON,
        )
    return evaluator


def prediction_frame(model, frame, evaluator):
    values = model.predict(frame)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Некорректные значения прогноза.")
    return pd.DataFrame(
        values.reshape(HORIZON, len(evaluator.series_index)).T,
        index=evaluator.series_index,
        columns=evaluator.forecast_dates,
    )


### 4.1. Проверка подготовки

Готовим признаки для одного будущего дня и проверяем 67 столбцов, порядок рядов, категории и размер плана. Полную обучающую таблицу здесь не создаём. Этот блок не запускает модель.


In [ ]:
check_features = make_daily_features_final(
    CONTROL_ORIGIN, CONTROL_ORIGIN + pd.Timedelta(days=1)
)
training_plan = make_training_plan(CONTROL_ORIGIN, HORIZONS_PER_ORIGIN)
if check_features.columns.tolist() != FEATURE_COLUMNS or len(FEATURE_COLUMNS) != 67:
    raise ValueError("Изменился состав итоговых признаков.")
if not check_features.index.equals(wrmsse_evaluator.series_index):
    raise ValueError("Изменился порядок товаров и магазинов.")
if daily_history_dates[-1] != PUBLIC_ORIGIN:
    raise ValueError("Изменился конец доступной истории.")
if len(training_plan) * len(check_features) != 25_367_680:
    raise ValueError("Изменился размер обучающего плана.")
if training_plan.duplicated(["origin", "horizon"]).any():
    raise ValueError("В плане есть повторяющиеся примеры.")
print("Признаки:", check_features.shape)
print("Пар дата — горизонт:", len(training_plan))
print("Обучающих строк:", f"{len(training_plan) * len(check_features):,}")
display(check_features.loc[:, [*CATEGORICAL_FEATURES, *TARGET_PRICE_FEATURES]].head())
del check_features
gc.collect()


## 5. Результаты выполненных экспериментов

Следующие таблицы составлены по сохранённым протоколам и выводу Kaggle. Они не пересчитываются при чтении ноутбука. Все поздние опыты с плотностью и новыми признаками относятся к **28 марта — 24 апреля 2016 года**; они ещё не подтверждены на других периодах.

### 5.1. CatBoost и первый LightGBM

| Модель | Исторический WRMSSE | Public WRMSSE | Комментарий |
|---|---:|---:|---|
| CatBoost после локального Optuna | 0.56683 | 0.63673 | 63 признака, 52 × 2, 768 деревьев |
| Прежний перенос CatBoost на Kaggle | 0.56605 | 0.63994 | До исправления Bottleneck |
| LightGBM, остановка по Tweedie | 0.95840 | — | Выбраны 52 дерева |
| LightGBM, остановка по WRMSSE | 0.55498 | 0.67879 | Выбраны 386 деревьев |

Tweedie остаётся функцией потерь. Критерий выбора числа деревьев заменили на WRMSSE, поскольку ранний минимум Tweedie давал плохой результат по целевой метрике. Историческое окно использовалось для этого выбора. На public первый LightGBM уступил CatBoost.

В двух проверенных срезах подготовки CatBoost расхождение с локальной версией исчезло после отключения Bottleneck. CatBoost с одинаковой подготовкой и настройками повторно не обучали; влияние исправления на его скор пока не измерено.

### 5.2. Плотность обучающей выборки

| Модель | План | Признаков | WRMSSE | Bias | Время fit, мин. |
|---|---|---:|---:|---:|---:|
| CatBoost | 52 × 2 | 63 | 0.56683 | −1.66% | — |
| CatBoost | 52 × 4 | 63 | 0.57435 | −1.87% | — |
| LightGBM | 52 × 2 | 63 | 0.55498 | −2.20% | — |
| LightGBM | 52 × 4 | 63 | 0.55332 | −2.03% | — |
| LightGBM | 52 × 8 | 63 | 0.53828 | −0.93% | 12.0 |
| LightGBM | 52 × 16 | 63 | 0.53362 | −1.38% | 23.2 |

У CatBoost 52 × 4 хуже локального контроля 52 × 2, поэтому его плотность пока не увеличиваем. Для LightGBM переходим к 52 × 16: WRMSSE ниже исходного 52 × 2 на 3.85%. Результат одного окна ещё не определяет лучшую плотность для остальных периодов.

Время полного опыта 52 × 4 — 40.6 минуты для CatBoost и 6.3 для LightGBM; это не чистое время fit, поэтому в колонке fit оно не смешивается с последующими измерениями.

### 5.3. Цена целевой недели и отклонённые гипотезы

| Вариант LightGBM, 52 × 16 | Признаков | WRMSSE | Bias | Решение |
|---|---:|---:|---:|---|
| Без цены целевой недели | 63 | 0.53362 | −1.38% | Исходный контроль |
| Цена целевой недели | **67** | **0.53079** | **−1.21%** | **Оставляем** |
| Цена + EWMA | 68 | 0.53081 | −1.26% | Не добавляем |
| Цена + лаги 7, 14, 21 | 70 | 0.53230 | −1.34% | Не добавляем |

Цена улучшила WRMSSE примерно на 0.53% относительно 63 признаков. EWMA практически ничего не изменил. Три коротких лага ухудшили скор примерно на 0.29% относительно ценовой модели. Эти выводы относятся к проверенным настройкам и периоду.

Для плотности, цены и EWMA в checkpoint есть точные JSON-протоколы. Для коротких лагов в предоставленных файлах доступен только сохранённый вывод ноутбука: WRMSSE, Bias и gain округлены. Полный протокол и модель этого опыта не представлены; точные числа не восстанавливались из округлённой таблицы.

Повторяющиеся блоки обучения отклонённых вариантов удалены из рабочего пути. Отдельные GPU-диагностики и установка CUDA для CPU-контроля не нужны. Успешный GPU-бенчмарк не получен, поэтому ускорение и качество GPU здесь не заявляются.


## 6. Повторное обучение принятого CPU-контроля

Один запуск: 52 × 16, 67 признаков, 386 деревьев. План создаётся из дат; проходить старые опыты плотности и подключать их модели не требуется. Функция сохраняет модель, прогноз, метрики, важности признаков, план и протокол в новую папку `control_67_features_cpu_v1`.

Ранняя остановка здесь выключена: воспроизводим фиксированный контроль. В следующем поиске Optuna число деревьев будем выбирать по WRMSSE с большим верхним лимитом. Нынешние 386 деревьев были выбраны раньше, на 52 × 2.

При оформлении этот код не выполнялся. Сохранённый WRMSSE 0.53079 относится к исходному запуску `target_week_price_52x16_v1`, а не к новой функции после перестановки кода.


In [ ]:
def train_cpu_control():
    versions = {
        "python": platform.python_version(),
        "lightgbm": lgb.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }
    for library, expected in EXPECTED_LIBRARIES.items():
        if versions[library] != expected:
            raise RuntimeError(
                f"Для повторения контроля нужен {library}=={expected}; "
                f"сейчас {versions[library]}."
            )
    if pd.get_option("compute.use_bottleneck"):
        raise RuntimeError("Перед подготовкой признаков отключи Bottleneck.")
    clear_feature_caches()
    plan = make_training_plan(CONTROL_ORIGIN, HORIZONS_PER_ORIGIN)
    evaluator = make_window_evaluator(CONTROL_ORIGIN)
    if not daily_history.index.equals(evaluator.series_index):
        raise ValueError("Изменился порядок рядов.")
    results = RESULTS / "control_67_features_cpu_v1" / str(CONTROL_ORIGIN.date())
    models = MODELS / "control_67_features_cpu_v1" / str(CONTROL_ORIGIN.date())
    results.mkdir(parents=True, exist_ok=True)
    models.mkdir(parents=True, exist_ok=True)
    plan.to_csv(results / "training_plan.csv", index=False, date_format="%Y-%m-%d")
    started = perf_counter()
    pairs = plan[["origin", "target_date"]].itertuples(index=False, name=None)
    X_train, y_train = build_feature_dataset(pairs, label_cutoff=CONTROL_ORIGIN)
    valid_pairs = [(CONTROL_ORIGIN, date) for date in evaluator.forecast_dates]
    X_valid, _ = build_feature_dataset(valid_pairs)
    actual = daily_history.loc[evaluator.series_index, evaluator.forecast_dates]
    if len(X_train) != 25_367_680 or len(FEATURE_COLUMNS) != 67:
        raise ValueError("Изменился размер контрольной выборки.")
    data_seconds = perf_counter() - started
    table_gib = X_train.memory_usage(index=True, deep=True).sum() / 1024**3
    print(f"Строк: {len(X_train):,}; признаков: {len(FEATURE_COLUMNS)}")
    print(f"Таблица: {table_gib:.2f} GiB; подготовка: {data_seconds / 60:.1f} мин.")
    fit_started = perf_counter()

    def progress(env):
        completed = env.iteration + 1
        if completed % 100 == 0 or completed == env.end_iteration:
            print(
                f"CPU: {completed}/{MODEL_PARAMETERS['n_estimators']} деревьев, "
                f"{(perf_counter() - fit_started) / 60:.1f} мин.",
                flush=True,
            )

    model = lgb.LGBMRegressor(**MODEL_PARAMETERS)
    model.fit(
        X_train,
        y_train,
        categorical_feature=CATEGORICAL_FEATURES,
        callbacks=[progress],
    )
    fit_seconds = perf_counter() - fit_started
    del X_train, y_train
    gc.collect()
    prediction = prediction_frame(model, X_valid, evaluator)
    metrics, levels = price_evaluate(evaluator, actual, prediction)
    model.booster_.save_model(str(models / "lightgbm.txt"))
    prediction.to_csv(results / "predictions.csv.gz", compression="gzip")
    levels.to_csv(results / "metrics_by_level.csv")
    pd.DataFrame([metrics]).to_csv(results / "metrics.csv", index=False)
    pd.Series(
        model.booster_.feature_importance(importance_type="gain"),
        index=FEATURE_COLUMNS,
        name="importance_gain",
    ).sort_values(ascending=False).to_csv(results / "feature_importance_gain.csv")
    protocol = {
        "origin": str(CONTROL_ORIGIN.date()),
        "training_origins": N_ORIGINS,
        "horizons_per_origin": HORIZONS_PER_ORIGIN,
        "train_rows": len(plan) * len(evaluator.series_index),
        "features": FEATURE_COLUMNS,
        "categorical_features": CATEGORICAL_FEATURES,
        "params": model.get_params(),
        "actual_trees": int(model.booster_.current_iteration()),
        "early_stopping": False,
        "compute_use_bottleneck": False,
        "target_price_source": "provided sell_prices.csv, target week",
        "historical_price_statistics": "13 weeks ending at forecast origin",
        "versions": versions,
        "data_seconds": data_seconds,
        "fit_seconds": fit_seconds,
        "training_table_gib": float(table_gib),
        "metrics": metrics,
    }
    (results / "protocol.json").write_text(
        json.dumps(protocol, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    comparison = pd.DataFrame(
        [SAVED_CONTROL_METRICS, metrics],
        index=["Сохранённый контроль", "Новый CPU-запуск"],
    )
    display(comparison.style.format({"WRMSSE": "{:.5f}", "Bias": "{:+.2%}"}))
    comparison.to_csv(results / "comparison.csv")
    archive = PROJECT_ROOT.parent / "demand-planner_control_67_cpu.zip"
    temporary = archive.with_suffix(".tmp.zip")
    with ZipFile(temporary, "w", ZIP_DEFLATED, compresslevel=1) as output:
        for directory in (results, models):
            for path in sorted(directory.rglob("*")):
                if path.is_file():
                    output.write(path, path.relative_to(PROJECT_ROOT))
    temporary.replace(archive)
    print(f"Обучение: {fit_seconds / 60:.1f} мин.")
    display(FileLink(os.path.relpath(archive, Path.cwd())))
    clear_feature_caches()
    return comparison


### 6.1. Запуск по флагу

Чтобы повторить только итоговый контроль, переключаем `RUN_CONTROL_TRAINING` в первой ячейке на `True` и запускаем подготовку и этот блок. При `False` обучения нет. Для просмотра истории результатов достаточно сохранённых таблиц выше.


In [ ]:
if RUN_CONTROL_TRAINING:
    cpu_control_comparison = train_cpu_control()
else:
    print("CPU-контроль не перезапускался: RUN_CONTROL_TRAINING = False.")


## 7. Что дальше

Текущая отправная точка: **67 признаков, 52 × 16, CPU, WRMSSE 0.53079 на историческом окне**. EWMA и короткие лаги в неё не входят.

1. Подготовить отдельный `05_optuna_cpu.ipynb`: только нужные данные, признаки и поиск, чтобы `Save & Run All` не повторял историю экспериментов.
2. Подобрать параметры LightGBM, включая Tweedie; контрольную попытку выполнить с большим лимитом деревьев и остановкой по WRMSSE. Сохранять SQLite-базу, результаты завершённых попыток и лучшие модели.
3. Проверить двух финалистов и контроль на других исторических окнах, затем проверить финалистов на 52 × 28.
4. Проверить ансамбль с CatBoost и калибровочный множитель по историческим прогнозам. Сохранять вариант, который улучшает WRMSSE.
5. Зафиксировать настройки, подготовить итоговый M5-прогноз и оформить воспроизводимый запуск и результаты в GitHub.

Public-период уже просмотрен в ходе разработки. Его не называем нетронутым тестом. Официальный серверный score и место на Kaggle пока не получены.
